In [1]:
import pandas as pd
import re

def is_barred(morphological_type):
    """
    Determine if a galaxy is barred based on its morphological type.
    Returns 1 for barred (SB), 0 for unbarred (SA), or handles special cases.
    """
    if pd.isna(morphological_type) or morphological_type == '':
        return None  # Unknown classification
    
    morph = str(morphological_type).strip()
    
    # Check for barred indicators (SB)
    if morph.startswith('SB'):
        return 1
    # Check for SAB (weakly barred - treating as barred)
    elif morph.startswith('SAB'):
        return 1
    # Check for unbarred indicators (SA)
    elif morph.startswith('SA'):
        return 0
    # Check for S0 types (lenticular - typically unbarred unless specified)
    elif morph.startswith('S0'):
        return 0
    # Check for simple S classification (usually unbarred)
    elif morph.startswith('S') and not morph.startswith('SB'):
        return 0
    # Check for E (elliptical - no bar)
    elif morph.startswith('E'):
        return 0
    # Check for (R) prefix cases - look at what follows
    elif morph.startswith('(R'):
        # Extract the part after (R) or similar notations
        match = re.search(r'\(R[^\)]*\)\s*(.+)', morph)
        if match:
            core_type = match.group(1)
            if core_type.startswith('SB'):
                return 1
            elif core_type.startswith('SAB'):
                return 1
            elif core_type.startswith('SA'):
                return 0
    
    # Default: unable to determine
    return None

def process_file(input_file, output_file):
    """
    Process a CSV file and create a label file with Target Name and Label.
    """
    # Read the CSV file
    df = pd.read_csv(input_file)
    
    # Create labels based on morphological type
    df['Label'] = df['Morphological Type'].apply(is_barred)
    
    # Create output dataframe with only Target Name and Label
    output_df = df[['Target Name', 'Label']].copy()
    
    # Optionally: drop rows where Label is None (unknown classification)
    # Uncomment the next line if you want to exclude galaxies with unclear classification
    # output_df = output_df.dropna(subset=['Label'])
    
    # Convert Label to integer (if not None)
    output_df['Label'] = output_df['Label'].astype('Int64')  # Int64 allows NaN values
    
    # Save to CSV
    output_df.to_csv(output_file, index=False)
    
    # Print statistics
    total = len(output_df)
    barred = (output_df['Label'] == 1).sum()
    unbarred = (output_df['Label'] == 0).sum()
    unknown = output_df['Label'].isna().sum()
    
    print(f"\n{input_file} -> {output_file}")
    print(f"Total galaxies: {total}")
    print(f"Barred (1): {barred} ({barred/total*100:.1f}%)")
    print(f"Unbarred (0): {unbarred} ({unbarred/total*100:.1f}%)")
    print(f"Unknown: {unknown} ({unknown/total*100:.1f}%)")
    
    return output_df

# Process both files
print("Processing CSV files...")
print("=" * 60)

train_labels = process_file('train.csv', 'trainlabel.csv')
test_labels = process_file('test.csv', 'testlabel.csv')

print("\n" + "=" * 60)
print("Processing complete!")
print("\nFiles created:")
print("  - trainlabel.csv")
print("  - testlabel.csv")

# Show a few examples from each file
print("\n" + "=" * 60)
print("\nSample from trainlabel.csv:")
print(train_labels.head(10))

print("\nSample from testlabel.csv:")
print(test_labels.head(10))

Processing CSV files...

train.csv -> trainlabel.csv
Total galaxies: 466
Barred (1): 251 (53.9%)
Unbarred (0): 213 (45.7%)
Unknown: 2 (0.4%)

test.csv -> testlabel.csv
Total galaxies: 117
Barred (1): 69 (59.0%)
Unbarred (0): 48 (41.0%)
Unknown: 0 (0.0%)

Processing complete!

Files created:
  - trainlabel.csv
  - testlabel.csv


Sample from trainlabel.csv:
  Target Name  Label
0    NGC 0024      0
1    NGC 0099      0
2    NGC 0115      1
3    NGC 0151      1
4    NGC 0165      1
5   PGC 02269      1
6    NGC 0195      1
7    NGC 0213      1
8    NGC 0253      1
9   NGC 0247B      1

Sample from testlabel.csv:
    Target Name  Label
0      NGC 0055      1
1      NGC 0131      1
2     UGC 00344      1
3      NGC 0223      1
4   MESSIER 031      0
5     UGC 00484      1
6      NGC 0247      1
7     UGC 00533      0
8  ESO 296-G002      1
9       IC 1698      0
